<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Chapter 4: Implementing a GPT model from Scratch to Generate Text

In [1]:
from importlib.metadata import version

modules = [
    "matplotlib",
    "torch",
    "tiktoken"
]

for module in modules:
    print(f"{module} vesion: {version(module)}")


matplotlib vesion: 3.10.8
torch vesion: 2.10.0
tiktoken vesion: 0.12.0


In this chapter, we will implement GPT-like LLM Architecture; the next chapter will focus on training this LLM

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/01.webp" width="500px">

## 4.1 Coding an LLM architecture

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/02.webp" width="400px">

In [2]:
# Model configurations

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/03.webp" width="500px">

In [3]:
# Backbone

import torch
import torch.nn as nn

In [4]:
class DummyGPT(nn.Module):
    """Dummy GPT Model Architecture"""

    def __init__(self, cfg):
        super().__init__()

        # Embedding layer
        self.tok_emb = nn.Embedding(
            num_embeddings=cfg["vocab_size"], 
            embedding_dim=cfg["emb_dim"]
        )

        # Positional embedding layer
        self.pos_emb = nn.Embedding(
            num_embeddings=cfg["context_length"],
            embedding_dim=cfg["emb_dim"]
        )

        # Dropout layer
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Placehold for transformer block
        self.trf_blocks = nn.Sequential(
            *[DummyTransferBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Placeholder for layer norm
        self.final_norm = DummyLayerNorm(
            cfg["emb_dim"]
        )
        self.out_head = nn.Linear(
            in_features=cfg["emb_dim"],
            out_features=cfg["vocab_size"],
            bias=False
        )

    def forward(self, in_idx):
        """Forward method"""

        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(
                seq_len,
                device=in_idx.device
            )
        )

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


# Dummy transformer block
class DummyTransferBlock(nn.Module):
    """Dummy transformer block"""

    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        """Forward method"""

        return x
    
# Dummy layer normalization
class DummyLayerNorm(nn.Module):
    """Dummy Layer normalization"""

    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        """Forward method"""
        return x

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/04.webp?123" width="500px">

In [5]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [6]:
batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

In [7]:
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

batch

[tensor([6109, 3626, 6100,  345]), tensor([6109, 1110, 6622,  257])]

In [8]:
batch = torch.stack(batch, dim=0)
batch

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

In [9]:
torch.manual_seed(123)

model = DummyGPT(GPT_CONFIG_124M)
model

DummyGPT(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): DummyTransferBlock()
    (1): DummyTransferBlock()
    (2): DummyTransferBlock()
    (3): DummyTransferBlock()
    (4): DummyTransferBlock()
    (5): DummyTransferBlock()
    (6): DummyTransferBlock()
    (7): DummyTransferBlock()
    (8): DummyTransferBlock()
    (9): DummyTransferBlock()
    (10): DummyTransferBlock()
    (11): DummyTransferBlock()
  )
  (final_norm): DummyLayerNorm()
  (out_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [10]:
logits = model(batch)

print("Output shape: ", logits.shape)
print(logits)

Output shape:  torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


## 4.2 Normalizing activations with layer normalization

- Layer normalization, also known as LayerNorm ([Ba et al. 2016](https://arxiv.org/abs/1607.06450)), centers the activations of a neural network layer around a mean of 0 and normalizes their variance to 1
- This stabilizes training and enables faster convergence to effective weights
- Layer normalization is applied both before and after the multi-head attention module within the transformer block, which we will implement later; it's also applied before the final output layer

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/05.webp" width="400px">

In [11]:
# Let's see how layer norm works internally

torch.manual_seed(123)

# create 2 training examples with 5 dimensions (features) each
batch_example = torch.randn(2, 5)
batch_example

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])

In [12]:
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


- Let's compute the mean and variance for each of the 2 inputs above:

In [13]:
print(out.mean())
print(out.mean(dim=-1))
print(out.mean(dim=-1, keepdim=True))

tensor(0.1747, grad_fn=<MeanBackward0>)
tensor([0.1324, 0.2170], grad_fn=<MeanBackward1>)
tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)


In [16]:
print(out.var())
print(out.var(dim=-1))
print(out.var(dim=-1, keepdim=True))

tensor(0.0305, grad_fn=<VarBackward0>)
tensor([0.0231, 0.0398], grad_fn=<VarBackward0>)
tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [17]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/06.webp" width="400px">

In [18]:
out_norm = (out - mean) / torch.sqrt(var)
print("Normalized layer outputs: \n", out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

Normalized layer outputs: 
 tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean:
 tensor([[-5.9605e-08],
        [ 1.9868e-08]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


- Each input is centered at 0 and has a unit variance of 1; to improve readability, we can disable PyTorch's scientific notation:

In [19]:
torch.set_printoptions(sci_mode=False)

print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[-0.0000],
        [ 0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [20]:
class LayerNorm(nn.Module):
    """Layer normalization"""
    
    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        """Forward method"""

        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift
    

**Scale and shift**

- Note that in addition to performing the normalization by subtracting the mean and dividing by the variance, we added two trainable parameters, a `scale` and a `shift` parameter
- The initial `scale` (multiplying by 1) and `shift` (adding 0) values don't have any effect; however, `scale` and `shift` are trainable parameters that the LLM automatically adjusts during training if it is determined that doing so would improve the model's performance on its training task
- This allows the model to learn appropriate scaling and shifting that best suit the data it is processing
- Note that we also add a smaller value (`eps`) before computing the square root of the variance; this is to avoid division-by-zero errors if the variance is 0

**Biased variance**
- In the variance calculation above, setting `unbiased=False` means using the formula $\frac{\sum_i (x_i - \bar{x})^2}{n}$ to compute the variance where n is the sample size (here, the number of features or columns); this formula does not include Bessel's correction (which uses `n-1` in the denominator), thus providing a biased estimate of the variance 
- For LLMs, where the embedding dimension `n` is very large, the difference between using n and `n-1`
 is negligible
- However, GPT-2 was trained with a biased variance in the normalization layers, which is why we also adopted this setting for compatibility reasons with the pretrained weights that we will load in later chapters

- Let's now try out `LayerNorm` in practice: